# AI Engineering MasterClass - Class 4 Lab
### Generation Control and Production Prompting

**By the end of this notebook you will have:**

1. Controlled a model's output with `max_tokens`, `stop` and `seed`, and found where each one breaks.
2. Watched free-text JSON break your code, then fixed it properly with a strict schema.
3. Validated every answer with Pydantic, and retried automatically when validation failed.
4. Built a tool-calling loop: the model asks, your code decides and runs, the model answers.
5. Kept your prompts in versioned files and tested version 2 against version 1 with numbers.

The running example is a support bot for an online shop in Bangladesh, in English and Bangla.
Everything runs on your free Groq tier. The whole notebook costs a fraction of a cent.

---
## Setup

In [35]:
import os, json, re, time
from pathlib import Path
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env"); break
API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("Grok_API_KEY")
assert API_KEY, "No key found. Put GROQ_API_KEY=... in a .env file here or in the folder above."

import requests
from groq import Groq
client = Groq(api_key=API_KEY)

models = requests.get("https://api.groq.com/openai/v1/models",
                      headers={"Authorization": f"Bearer {API_KEY}"}, timeout=30).json()["data"]
PRICES = {m["id"]: {"in": float(m["pricing"]["prompt"]) * 1e6, "out": float(m["pricing"]["completion"]) * 1e6}
          for m in models if (m.get("pricing") or {}).get("completion")}

FAST  = "openai/gpt-oss-20b"     # reasoning model
OTHER = "qwen/qwen3.8-27b"       # second model, for comparison
SPENT = {"usd": 0.0, "calls": 0}

def call(model=FAST, **kw):
    """One API call that also keeps a running cost total (Class 1 habit)."""
    if model.startswith("openai/"):
        kw.setdefault("reasoning_effort", "low")
    kw.setdefault("max_tokens", 600)
    r = client.chat.completions.create(model=model, **kw)
    p = PRICES[model]
    SPENT["usd"] += r.usage.prompt_tokens / 1e6 * p["in"] + r.usage.completion_tokens / 1e6 * p["out"]
    SPENT["calls"] += 1
    return r

print("Ready. Models:", FAST, "and", OTHER)

Ready. Models: openai/gpt-oss-20b and qwen/qwen3.8-27b


---
## Step 1 - The controls you set on every call

| Control | What it does |
|---|---|
| `temperature` | How adventurous the next-token pick is (Class 3) |
| `max_tokens` | A hard ceiling on output tokens, thinking included |
| `stop` | Strings that end generation the moment they appear |
| `seed` | Asks for the same random choices on every run |

Each one has a trap. Let us find them.

### `stop`: works on one model, empties the other

The prompt asks for three cities, the word END, then three more. We set `stop=["END"]`,
expecting just the first three.

In [36]:
prompt = "List three Bangladeshi cities, one per line, then write the word END on its own line, then list three more."

for model in (OTHER, FAST):
    r = call(model=model, temperature=0, stop=["END"], messages=[{"role": "user", "content": prompt}])
    print(f"{model:22} output tokens {r.usage.completion_tokens:3}   content: {r.choices[0].message.content!r}")

qwen/qwen3.8-27b       output tokens  27   content: 'Dhaka\nChattogram\nSylhet\n'
openai/gpt-oss-20b     output tokens  49   content: ''


Qwen stopped where we asked. gpt-oss returned **an empty string**, and we still paid for its tokens.

Why: gpt-oss is a reasoning model. It *thinks* before answering, and while planning it wrote
the word END. The stop sequence fired inside the hidden thinking, before any answer existed.

**Rule:** do not use `stop` with reasoning models. Constrain the format instead (Step 3).

### `seed`: reproducible-ish

Same creative prompt at temperature 1.2, four times with a seed and four times without.

In [37]:
prompt = "Invent a name for a Dhaka tea stall. Reply with the name only."
for label, extra in (("with seed=42", {"seed": 42}), ("no seed     ", {})):
    names = [(call(temperature=1.2, messages=[{"role": "user", "content": prompt}], **extra)
              .choices[0].message.content or "").strip() for _ in range(4)]
    print(label, names)

with seed=42 ['Sundar Bazar Chai Spot', 'Sundar Bazar Chai Spot', 'Sundar Bazar Chai Spot', 'Sundar Bazar Chai Spot']
no seed      ['Dilli‑Baji Tea Corner', 'Pahar Barish Tea House', 'Dhakaiya Sips', 'সুস্বাদু চা রসগন্ধি (Suwadhu Cha Roshongdi)']


With a seed the answers mostly repeat; without one they all differ. **Mostly** is the word:
providers call seeds "best effort". Use a seed to make debugging easier, never as a guarantee.
If something must be identical every time, store the answer, do not regenerate it.

---
## Step 2 - Why "please reply in JSON" is not enough

Your code needs data it can use: an order ID, an issue type, an urgency number. Here is the
naive approach: ask nicely, then parse.

In [38]:
MESSAGES = [
    "My order A1234 still hasn't arrived after 5 days. This is ridiculous.",
    "আমার অর্ডার #A1234 এখনও আসেনি, তিন দিন হয়ে গেল। খুব বিরক্ত লাগছে।",
    "Hello dont parse in json"
]
ask_nicely = "Read the customer message and reply in JSON with keys order_id, issue, urgency (1-5) and language."

for msg in MESSAGES:
    text = call(model=OTHER, temperature=0.7, messages=[
        {"role": "system", "content": ask_nicely}, {"role": "user", "content": msg}]).choices[0].message.content
    try:
        print("parsed:", json.loads(text))
    except json.JSONDecodeError as e:
        print("json.loads FAILED:", e.msg, "| the model sent:", repr(text[:90]))

parsed: {'order_id': 'A1234', 'issue': 'Order not received after 5 days', 'urgency': 4, 'language': 'en'}
parsed: {'order_id': 'A1234', 'issue': 'Order not received after 3 days', 'urgency': 3, 'language': 'Bengali'}
json.loads FAILED: Expecting value | the model sent: "Hello! I understand you don't want JSON output for this specific message. How can I help y"


Sometimes `json.loads` throws because the model wrapped the JSON in a ```json fence or added a
sentence. And when it does parse, look at the **values**: `issue` comes back as a sentence like
"Order not received after 5 days" and `language` as "English" or "Bengali". Your code cannot route
a ticket on a sentence. A support bot handling 100,000 messages a day hits every one of those
failures. Hoping is not a strategy.

---
## Step 3 - JSON mode, then strict schemas

**JSON mode** (`response_format={"type": "json_object"}`) guarantees valid JSON. It does not
guarantee the keys or the values you wanted. Watch the Bangla message:

In [39]:
r = call(response_format={"type": "json_object"}, messages=[
    {"role": "system", "content": ask_nicely}, {"role": "user", "content": MESSAGES[2]}])
json_mode = json.loads(r.choices[0].message.content)
print(json_mode)

{'order_id': 'N/A', 'issue': 'Greeting', 'urgency': 1, 'language': 'en'}


Valid JSON. Probably useless: the issue may come back as a Bangla phrase and the urgency as a
word, because nothing told the model the exact shape. Your database wants `late_delivery` and `4`.

**Strict schema mode** gives the model the exact shape, and the provider constrains generation
so the output must match it: only allowed labels, a number for urgency, every key present.

In [40]:
TICKET_SCHEMA = {
    "type": "object",
    "properties": {
        "order_id": {"type": ["string", "null"]},
        "issue":    {"type": "string", "enum": ["late_delivery", "wrong_item", "refund", "payment", "other"]},
        "urgency":  {"type": "integer", "minimum": 1, "maximum": 5},
        "language": {"type": "string", "enum": ["bn", "en", "mixed"]},
    },
    "required": ["order_id", "issue", "urgency", "language"],
    "additionalProperties": False,
}
STRICT = {"type": "json_schema", "json_schema": {"name": "ticket", "strict": True, "schema": TICKET_SCHEMA}}

for msg in MESSAGES:
    r = call(response_format=STRICT, messages=[
        {"role": "system", "content": "Extract the support ticket."}, {"role": "user", "content": msg}])
    print(r.choices[0].message.content)

{"order_id":"A1234","issue":"late_delivery","urgency":5,"language":"en"}
{"order_id":"A1234","issue":"late_delivery","urgency":5,"language":"bn"}
{"order_id":null,"issue":"other","urgency":1,"language":"en"}


Same labels, same types, every time, in both languages.

### Never trust, always validate: Pydantic

Even with a schema, validate in your own code. Providers have bugs, schemas change, and
some models only support JSON mode. Pydantic turns "looks right" into "is right, or raises":

In [41]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError

class Ticket(BaseModel):
    order_id: Optional[str]
    issue: Literal["late_delivery", "wrong_item", "refund", "payment", "other"]
    urgency: int = Field(ge=1, le=5)
    language: Literal["bn", "en", "mixed"]

try:
    Ticket.model_validate(json_mode)
    print("JSON-mode answer passed validation this time.")
except ValidationError as e:
    print(f"JSON-mode answer REJECTED ({e.error_count()} problems):")
    for err in e.errors():
        print("  ", err["loc"], "->", err["msg"])

JSON-mode answer REJECTED (1 problems):
   ('issue',) -> Input should be 'late_delivery', 'wrong_item', 'refund', 'payment' or 'other'


### The production pattern: validate, and retry with the error

When validation fails, send the model its own mistake and ask again. One retry fixes most cases.

In [42]:
def extract_ticket(message, max_attempts=2):
    convo = [{"role": "system", "content": ask_nicely + " Use these issue labels only: late_delivery, wrong_item, refund, payment, other. urgency is an integer. language is bn, en or mixed."},
             {"role": "user", "content": message}]
    for attempt in range(1, max_attempts + 1):
        raw = call(response_format={"type": "json_object"}, messages=convo).choices[0].message.content
        try:
            return Ticket.model_validate_json(raw), attempt
        except ValidationError as e:
            convo += [{"role": "assistant", "content": raw},
                      {"role": "user", "content": f"That failed validation: {e.errors()[0]['msg']} at {e.errors()[0]['loc']}. Fix it and return only the JSON."}]
    raise RuntimeError("Still invalid after retries. Send to a human.")

ticket, attempts = extract_ticket(MESSAGES[1])
print(ticket, f"   (attempts: {attempts})")

order_id='A1234' issue='late_delivery' urgency=5 language='bn'    (attempts: 1)


If it printed `attempts: 1`, validation passed first time because the prompt listed the labels.
The retry is insurance for the day it does not. Notice the last line of the function: when retries
run out, a human gets the ticket.
**A production system always has a way to fail safely.**

---
## Step 4 - Tool calling: the model asks, your code decides

The model cannot see your database. With **tool calling** you describe functions it may ask
for. The model replies "please call `get_order_status` with A1234". **Your code** checks the
request, runs the function, and sends the result back. Then the model writes the answer.

The model never runs anything. It only asks. You stay in control.

In [43]:
ORDERS = {  # pretend database
    "A1234": {"status": "in transit", "eta": "12 November", "courier": "Pathao"},
    "B7781": {"status": "delivered", "delivered_on": "6 November"},
    "C2020": {"status": "processing", "eta": "15 November"},
}

def get_order_status(order_id: str) -> dict:
    # Your code is the gatekeeper: validate everything the model asks for.
    if not re.fullmatch(r"[A-Z]\d{4}", order_id or ""):
        return {"error": "invalid order id format"}
    return ORDERS.get(order_id, {"error": "order not found"})

TOOLS = [{"type": "function", "function": {
    "name": "get_order_status",
    "description": "Look up the delivery status of one order by its ID, for example A1234.",
    "parameters": {"type": "object",
                   "properties": {"order_id": {"type": "string", "description": "Order ID, one capital letter and four digits"}},
                   "required": ["order_id"]}}}]
REGISTRY = {"get_order_status": get_order_status}

SYSTEM = ("You are a polite support agent for an online shop in Bangladesh. "
          "Reply in the same language the customer used in their last message: English if they wrote English, "
          "Bangla if they wrote Bangla. Use the tools to check facts; never guess an order status.")

def support_bot(question, max_turns=4, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for turn in range(max_turns):
        r = call(messages=messages, tools=TOOLS, tool_choice="auto")
        msg = r.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            fn = REGISTRY.get(tc.function.name)
            result = fn(**args) if fn else {"error": "unknown tool"}
            if verbose:
                print(f"   model asked: {tc.function.name}({args})  ->  {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result, ensure_ascii=False)})
    return "Sorry, I could not finish that. A human agent will follow up."

for q in ["Where is my order A1234?",
          "আমার অর্ডার B7781 কোথায়? পৌঁছেছে কি?",
          "Check order Z99 for me"]:
    print("Q:", q)
    print("A:", support_bot(q), "\n")

Q: Where is my order A1234?
   model asked: get_order_status({'order_id': 'A1234'})  ->  {'status': 'in transit', 'eta': '12 November', 'courier': 'Pathao'}
A: Your order A1234 is currently **in transit**. It is expected to arrive on **12 November** via **Pathao** courier. If you need any further assistance, please let us know! 

Q: আমার অর্ডার B7781 কোথায়? পৌঁছেছে কি?
   model asked: get_order_status({'order_id': 'B7781'})  ->  {'status': 'delivered', 'delivered_on': '6 November'}
A: অর্ডার B7781 সফলভাবে ডেলিভার হয়েছে এবং ৬ নভেম্বর আপনাকে পৌঁছে দেওয়া হয়েছে। যদি প্যাকেজে কোনো সমস্যা থাকে বা অন্য কোনো সহায়তা প্রয়োজন হয়, দয়া করে আমাদের জানান। ধন্যবাদ! 

Q: Check order Z99 for me
A: I’m sorry, but order IDs in our system must be in the format of one capital letter followed by four digits (e.g., A1234). The ID “Z99” doesn’t match that format, so I can’t look up its status. Could you double‑check the order number and let me know the correct ID? That way I can retrieve the delivery informat

Check the reply languages. An earlier version of this prompt said only "answer in the customer's
language" and the model replied to English questions in Bangla, probably because the prompt mentions
Bangladesh. Spelling the rule out fixed it. Vague instructions get interpreted; precise ones get followed.

Read the printed lines starting with **model asked**. That is the whole trick: the model chose
which function and which arguments; your code checked them and produced the facts.

Look at the last question. `Z99` is not a valid ID, so your function refused it. The model never
touched the database directly, so a strange request cannot do strange things.
This is the seed of agents (Classes 12 to 16) and of security (Class 17).

---
## Step 5 - Prompts are code: version them, test them

Your prompts live in files, in git, next to your code: `prompts/triage_v1.md` and
`prompts/triage_v2.md`. Open them in VS Code and compare.

- **v1** is one line of instructions.
- **v2** defines every label, gives rules for tricky cases (refund vs wrong item, Bangla digits
  in order numbers), and one Bangla example.

Is v2 actually better? Do not argue. Test. Here are fourteen real-looking messages with known
answers. The first eight are general. The last six are decided by **our shop's policy**: is a
broken item a refund or a wrong item? Is a rider asking for extra cash a payment problem?

In [44]:
TEST_SET = [
    ("My order A1234 still hasn't arrived after 5 days.",                        "late_delivery", "A1234"),
    ("আমার অর্ডার #B7781 এখনও আসেনি।",                                            "late_delivery", "B7781"),
    ("I got a blue shirt but I ordered black. Order C2020.",                      "wrong_item",    "C2020"),
    ("The product arrived broken, I want my money back. Order D5501",            "refund",        "D5501"),
    ("bKash থেকে টাকা দুইবার কেটেছে, অর্ডার E3003",                                "payment",       "E3003"),
    ("Can I change my delivery address?",                                         "other",         None),
    ("অর্ডার নম্বর ৪৫৬৭, ভুল সাইজ পাঠিয়েছেন।",                                     "wrong_item",    "4567"),
    ("Order H4545 was supposed to come yesterday, ki obostha?",                   "late_delivery", "H4545"),
    # --- cases decided by OUR shop policy, which no model can guess ---
    ("The phone arrived with a cracked screen. Please send a new one. Order J3141", "wrong_item",  "J3141"),
    ("I ordered 3 packets of tea but only 2 came. Order K2718",                   "wrong_item",    "K2718"),
    ("ডেলিভারি ম্যান অতিরিক্ত ৫০ টাকা চাইলো। অর্ডার L1618",                          "payment",       "L1618"),
    ("When will the refund for order m4040 reach my bKash?",                      "refund",        "M4040"),
    ("Order N5050 came damaged, the box was crushed.",                            "wrong_item",    "N5050"),
    ("অর্ডার P6060 এর রিফান্ড কবে পাবো? এক সপ্তাহ আগে চেয়েছি।",                       "refund",        "P6060"),
]

def run_prompt(version):
    system = Path(f"prompts/triage_{version}.md").read_text(encoding="utf-8")
    rows = []
    for msg, want_issue, want_id in TEST_SET:
        r = call(response_format=STRICT, temperature=0,
                 messages=[{"role": "system", "content": system}, {"role": "user", "content": msg}])
        t = Ticket.model_validate_json(r.choices[0].message.content)
        oid = (t.order_id or "").lstrip("#") or None
        rows.append({"message": msg, "want": [want_issue, want_id], "got": [t.issue, oid],
                     "correct": t.issue == want_issue and oid == want_id})
    return rows

RESULTS = {}
for v in ("v1", "v2"):
    RESULTS[v] = run_prompt(v)
    print(f"prompt {v}: {sum(r['correct'] for r in RESULTS[v])}/{len(TEST_SET)} correct")

prompt v1: 11/14 correct
prompt v2: 13/14 correct


In [45]:
print(f"{'message':44} {'v1':>22} {'v2':>22}")
for a, b in zip(RESULTS["v1"], RESULTS["v2"]):
    fmt = lambda r: ("ok" if r["correct"] else f"{r['got'][0]}/{r['got'][1]}")
    print(f"{a['message'][:43]:44} {fmt(a):>22} {fmt(b):>22}")

message                                                          v1                     v2
My order A1234 still hasn't arrived after 5                      ok                     ok
আমার অর্ডার #B7781 এখনও আসেনি।                                   ok                     ok
I got a blue shirt but I ordered black. Ord                      ok                     ok
The product arrived broken, I want my money                      ok                     ok
bKash থেকে টাকা দুইবার কেটেছে, অর্ডার E3003                      ok                     ok
Can I change my delivery address?                                ok                     ok
অর্ডার নম্বর ৪৫৬৭, ভুল সাইজ পাঠিয়েছেন।                          ok        wrong_item/None
Order H4545 was supposed to come yesterday,                      ok                     ok
The phone arrived with a cracked screen. Pl             other/J3141                     ok
I ordered 3 packets of tea but only 2 came.                      ok                     ok

**How to read this:**

- v1 probably handled the general cases well. That is the strict schema helping: the list of
  allowed labels is itself part of the prompt.
- Where v1 fails is the policy cases. No model can guess that **your** shop treats a broken item
  as `wrong_item`, or that order IDs are always upper case. That knowledge has to be written into
  the prompt. This is why prompts grow over time: they collect your business rules.
- Did v2 break anything v1 got right? That is called a **regression**. In our run it did: the
  new rule "order IDs are one capital letter and four digits" made v2 reject the Bangla-digit order
  number ৪৫৬৭, which has no letter, and return null. v1 had got it right. One new rule silently broke
  an old case. That is why you re-run the **whole** test set after every prompt change, not just
  the case you were fixing.
- Ten messages is a habit, not an evaluation. Class 6 builds a proper one.

**The workflow:** change the prompt file, run the tests, commit only if the score held or improved.
Exactly what you did with code in Class 2.

In [46]:
print(f"Whole notebook: {SPENT['calls']} API calls, ${SPENT['usd']:.5f}")
Path("class4_results.json").write_text(json.dumps({"prompt_tests": RESULTS, "spent_usd": SPENT["usd"]},
                                                  ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved class4_results.json")

Whole notebook: 51 API calls, $0.00279
Saved class4_results.json


---
## Before Class 5

1. Write `prompts/triage_v3.md` that passes all fourteen cases. If your v2 run showed the Bangla-digit
   regression, that is your starting point: older orders have IDs with four digits and no letter.
   Then add one new message v3 gets wrong, and fix it **without breaking any other case**.
2. Add a second tool, `request_refund(order_id, reason)`, that only accepts orders with status
   `delivered`. Show one call it accepts and one it refuses.
3. Push the notebook with outputs, your three prompt files and `class4_results.json`.

**Next class:** Class 5, Context Engineering. Context rot, compaction, caching, and what it all costs.

assignment solve base on given task

In [47]:
# Updated TEST_SET with 15 cases (14 old + 1 new)
TEST_SET = [
    ("My order A1234 still hasn't arrived after 5 days.",                        "late_delivery", "A1234"),
    ("আমার অর্ডার #B7781 এখনও আসেনি।",                                            "late_delivery", "B7781"),
    ("I got a blue shirt but I ordered black. Order C2020.",                      "wrong_item",    "C2020"),
    ("The product arrived broken, I want my money back. Order D5501",            "refund",        "D5501"),
    ("bKash থেকে টাকা দুইবার কেটেছে, অর্ডার E3003",                                "payment",       "E3003"),
    ("Can I change my delivery address?",                                         "other",         None),
    ("অর্ডার নম্বর ৪৫৬৭, ভুল সাইজ পাঠিয়েছেন।",                                     "wrong_item",    "4567"),
    ("Order H4545 was supposed to come yesterday, ki obostha?",                   "late_delivery", "H4545"),
    # --- cases decided by OUR shop policy ---
    ("The phone arrived with a cracked screen. Please send a new one. Order J3141", "wrong_item",  "J3141"),
    ("I ordered 3 packets of tea but only 2 came. Order K2718",                   "wrong_item",    "K2718"),
    ("ডেলিভারি ম্যান অতিরিক্ত ৫০ টাকা চাইলো। অর্ডার L1618",                          "payment",       "L1618"),
    ("When will the refund for order m4040 reach my bKash?",                      "refund",        "M4040"),
    ("Order N5050 came damaged, the box was crushed.",                            "wrong_item",    "N5050"),
    ("অর্ডার P6060 এর রিফান্ড কবে পাবো? এক সপ্তাহ আগে চেয়েছি।",                       "refund",        "P6060"),
    # --- New 15th case for v3 ---
    ("Rider was extremely rude and refused to come to the 4th floor. Order R8822", "other",        "R8822"),
]

RESULTS["v3"] = run_prompt("v3")
print(f"prompt v3: {sum(r['correct'] for r in RESULTS['v3'])}/{len(TEST_SET)} correct")

prompt v3: 15/15 correct


In [48]:
# class4_results.json ফাইলটিতে v1, v2 এবং নতুন v3 আপডেট করা
Path("class4_results.json").write_text(
    json.dumps({"prompt_tests": RESULTS, "spent_usd": SPENT["usd"]}, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Saved updated class4_results.json with v3!")

Saved updated class4_results.json with v3!


2.HOME TASK 2




In [49]:

def request_refund(order_id: str, reason: str) -> dict:
    order_id = (order_id or "").upper().lstrip("#")
    

    if not re.fullmatch(r"[A-Z]\d{4}|\d{4}", order_id):
        return {"error": "invalid order id format"}
        
    order = ORDERS.get(order_id)
    if not order:
        return {"error": "order not found"}
        
    
    if order.get("status") != "delivered":
        return {
            "status": "refused",
            "message": f"Refund refused. Order {order_id} is currently '{order.get('status')}'. Refunds can only be requested after the product is delivered."
        }
        
    
    return {
        "status": "accepted",
        "message": f"Refund request accepted for order {order_id}. Reason: '{reason}'. It will be processed soon."
    }

REFUND_TOOL = {
    "type": "function",
    "function": {
        "name": "request_refund",
        "description": "Request a refund for an order. Strictly requires the order status to be 'delivered'.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID, e.g. B7781 or A1234"},
                "reason": {"type": "string", "description": "Reason for asking refund"}
            },
            "required": ["order_id", "reason"]
        }
    }
}

TOOLS.append(REFUND_TOOL)
REGISTRY["request_refund"] = request_refund


print("=== Case 1: Accepted (Order B7781 is delivered) ===")
print("Customer: I received my order B7781 but it is damaged. Please process a refund.")
reply_1 = support_bot("I received my order B7781 but it is damaged. Please process a refund.")
print("Bot Reply:\n", reply_1)

print("\n" + "="*50 + "\n")

print("=== Case 2: Refused (Order A1234 is in transit) ===")
print("Customer: Can you refund my order A1234 right now?")
reply_2 = support_bot("Can you refund my order A1234 right now?")
print("Bot Reply:\n", reply_2)

=== Case 1: Accepted (Order B7781 is delivered) ===
Customer: I received my order B7781 but it is damaged. Please process a refund.
   model asked: get_order_status({'order_id': 'B7781'})  ->  {'status': 'delivered', 'delivered_on': '6 November'}
   model asked: request_refund({'order_id': 'B7781', 'reason': 'Damaged product received.'})  ->  {'status': 'accepted', 'message': "Refund request accepted for order B7781. Reason: 'Damaged product received.'. It will be processed soon."}
Bot Reply:
 Sure thing! I’ve submitted the refund request for order **B7781**.  
You’ll receive the refund soon. If you have any more questions or need further assistance, just let me know!


=== Case 2: Refused (Order A1234 is in transit) ===
Customer: Can you refund my order A1234 right now?
   model asked: get_order_status({'order_id': 'A1234'})  ->  {'status': 'in transit', 'eta': '12 November', 'courier': 'Pathao'}
Bot Reply:
 I’m sorry, but we can only process refunds for orders that have been **delive